# Model Visualization: SAM 2 vs Trained U-Net

This notebook visualizes the results of our teacher-student distillation experiment.
We load the **U-Net model trained on the compute node** and compare its predictions against **SAM 2 (Teacher)**.

In [ ]:
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import glob
import segmentation_models_pytorch as smp
from PIL import Image

# Configuration
DATA_DIR = "Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification"
MODEL_PATH = "unet_best.pth"  # This file will appear after training finishes
IMG_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

: 

In [ ]:
# Load SAM 2 (Ultralytics)
try:
    from ultralytics import SAM
    sam_model = SAM("sam2.1_b.pt")
    print("SAM 2 Loaded Successfully!")
except ImportError:
    print("Warning: Ultralytics SAM not found. Comparison will be limited to U-Net only.")
    sam_model = None

In [ ]:
# Load Trained U-Net
if not os.path.exists(MODEL_PATH):
    print(f"Error: Model file '{MODEL_PATH}' not found. Did the training job finish?")
else:
    unet = smp.Unet(
        encoder_name="resnet34", 
        encoder_weights=None, 
        in_channels=3, 
        classes=1
    )
    unet.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    unet.to(DEVICE)
    unet.eval()
    print("U-Net Loaded Successfully!")

In [ ]:
def visualize_comparison(img_path):
    image = Image.open(img_path).convert("RGB")
    
    # --- 1. SAM 2 Inference ---
    mask_sam = np.zeros((image.size[1], image.size[0]))
    sam_time = 0
    
    if sam_model:
        start_sam = time.time()
        # Run inference
        results = sam_model(img_path, verbose=False)
        sam_time = time.time() - start_sam
        
        # Extract mask
        if results and len(results) > 0 and results[0].masks is not None:
            masks = results[0].masks.data.cpu().numpy()
            # Start with empty mask
            mask_sam = np.any(masks, axis=0).astype(np.uint8)
            # Resize mask to image size if needed (Ultralytics usually returns original size)
            mask_sam = cv2.resize(mask_sam, (image.size[0], image.size[1]), interpolation=cv2.INTER_NEAREST)

    # --- 2. U-Net Inference ---
    # Preprocess
    unet_input = cv2.resize(np.array(image), (IMG_SIZE, IMG_SIZE))
    unet_input = unet_input / 255.0
    unet_input = np.transpose(unet_input, (2, 0, 1)).astype(np.float32)
    unet_input = torch.tensor(unet_input).unsqueeze(0).to(DEVICE)
    
    start_unet = time.time()
    with torch.no_grad():
        out_unet = unet(unet_input)
        out_unet = torch.sigmoid(out_unet).cpu().numpy()[0, 0]
    unet_time = time.time() - start_unet
    
    # --- Visualization ---
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    
    ax[0].imshow(image)
    ax[0].set_title("Input Image")
    ax[0].axis('off')
    
    ax[1].imshow(mask_sam, cmap='gray')
    ax[1].set_title(f"SAM 2 (Teacher)\nTime: {sam_time:.4f}s")
    ax[1].axis('off')
    
    ax[2].imshow(out_unet, cmap='magma')
    ax[2].set_title(f"U-Net (Student) Probability\nTime: {unet_time:.4f}s")
    ax[2].axis('off')

    # Thresholded U-Net
    ax[3].imshow(out_unet > 0.5, cmap='gray')
    ax[3].set_title("U-Net Binary Prediction")
    ax[3].axis('off')
    
    plt.show()

# Find test images (Try finding inside nested directories)
# Looking for 'fractured' cases specifically as they are more interesting
search_path = os.path.join(DATA_DIR, "**", "fractured", "*.jpg")
image_paths = glob.glob(search_path, recursive=True)

# Fallback if jpg not found
if not image_paths:
    search_path = os.path.join(DATA_DIR, "**", "fractured", "*.png")
    image_paths += glob.glob(search_path, recursive=True)
if not image_paths:
    # Just find any image
    search_path = os.path.join(DATA_DIR, "**", "*.jpg")
    image_paths += glob.glob(search_path, recursive=True)

print(f"Found {len(image_paths)} images for visualization.")

# Visualize Random Samples
import random
if image_paths:
    for _ in range(3):
        test_img = random.choice(image_paths)
        visualize_comparison(test_img)
else:
    print("No images found to visualize.")

In [ ]:
# Plot Training History (if available)
import pandas as pd

LOG_FILE = "training_log.txt"
if os.path.exists(LOG_FILE):
    try:
        df = pd.read_csv(LOG_FILE)
        plt.figure(figsize=(10, 5))
        plt.plot(df['Epoch'], df['Train_Loss'], label='Train Loss', marker='o')
        plt.plot(df['Epoch'], df['Val_Loss'], label='Validation Loss', marker='o')
        plt.title("Training Progress")
        plt.xlabel("Epoch")
        plt.ylabel("Dice Loss")
        plt.legend()
        plt.grid(True)
        plt.show()
    except Exception as e:
        print(f"Error reading log file: {e}")
else:
    print("Training log not found yet.")